# MatchAnything CLI JSON 可视化

这个 notebook 面向 `python -m matchanything match ... --output xxx.json` 生成的结果 JSON。

它会完成两件事：
1. 在 notebook 内直接调用 CLI 生成一个 `match.json`。
2. 读取 JSON，并复用仓库里的 `matchanything.ui.viz.draw_matches_core(...)` 做可视化。

你可以先直接运行默认配置；如果想测试错误提示，也可以把图片路径改成不存在的文件。

In [ ]:
from pathlib import Path

repo_root = Path('/Users/lanjie/Proj/3dgs/MatchAnything').resolve()
image0_path = repo_root / 'tests/data/02928139_3448003521.jpg'
image1_path = repo_root / 'tests/data/17295357_9106075285.jpg'
matcher = 'matchanything_eloftr'
output_json_path = repo_root / 'notebook/output/match_result.json'

print(f'repo_root: {repo_root}')
print(f'image0_path: {image0_path}')
print(f'image1_path: {image1_path}')
print(f'matcher: {matcher}')
print(f'output_json_path: {output_json_path}')


In [ ]:
import json
import os
import subprocess
import sys
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

src_root = repo_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from matchanything.ui.viz import draw_matches_core

plt.rcParams['figure.figsize'] = (10, 6)

def load_rgb_image(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'图片不存在: {path}')
    return np.array(Image.open(path).convert('RGB'))

def to_numpy_if_possible(value):
    if isinstance(value, dict):
        return {k: to_numpy_if_possible(v) for k, v in value.items()}
    if isinstance(value, list):
        try:
            arr = np.asarray(value)
        except Exception:
            return [to_numpy_if_possible(v) for v in value]
        if arr.dtype != object:
            return arr
        return [to_numpy_if_possible(v) for v in value]
    return value

def as_points(value):
    if value is None:
        return np.empty((0, 2), dtype=float)
    arr = np.asarray(value)
    if arr.size == 0:
        return np.empty((0, 2), dtype=float)
    return arr.reshape(-1, 2)

def as_confidence(value, expected_length):
    if value is None:
        return np.ones(expected_length, dtype=float)
    arr = np.asarray(value, dtype=float).reshape(-1)
    if arr.size == expected_length:
        return arr
    return np.ones(expected_length, dtype=float)

def count_rows(value):
    if value is None:
        return 0
    arr = np.asarray(value)
    if arr.size == 0:
        return 0
    return int(arr.shape[0])

def show_match_image(title, image):
    plt.figure(figsize=(14, 7))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

def format_array(value, precision=4):
    if value is None:
        return 'None'
    arr = np.asarray(value)
    if arr.size == 0:
        return '[]'
    return np.array2string(arr, precision=precision, suppress_small=True)


In [ ]:
for image_path in [image0_path, image1_path]:
    if not Path(image_path).exists():
        raise FileNotFoundError(f'图片路径不存在，请先修改配置单元: {image_path}')

output_json_path.parent.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    '-m',
    'matchanything',
    'match',
    str(image0_path),
    str(image1_path),
    '--matcher',
    matcher,
    '--output',
    str(output_json_path),
]

env = os.environ.copy()
existing_pythonpath = env.get('PYTHONPATH', '')
env['PYTHONPATH'] = str(src_root) if not existing_pythonpath else str(src_root) + os.pathsep + existing_pythonpath

print('即将执行命令:')
print(' '.join(command))

result = subprocess.run(
    command,
    cwd=repo_root,
    env=env,
    capture_output=True,
    text=True,
    check=False,
)

if result.stdout.strip():
    print('stdout:')
    print(result.stdout.strip())

if result.returncode != 0:
    raise RuntimeError('CLI 执行失败。\n\n' + (result.stderr.strip() or '<empty stderr>'))

print(f'CLI 执行成功，JSON 已写入: {output_json_path}')


In [ ]:
with output_json_path.open('r', encoding='utf-8') as f:
    raw_pred = json.load(f)

pred = {key: to_numpy_if_possible(value) for key, value in raw_pred.items()}
pred['image0_orig'] = load_rgb_image(image0_path)
pred['image1_orig'] = load_rgb_image(image1_path)

available_fields = sorted(raw_pred.keys())
raw_match_count = count_rows(pred.get('mkeypoints0_orig'))
ransac_match_count = count_rows(pred.get('mmkeypoints0_orig'))
has_homography = pred.get('H') is not None and np.asarray(pred.get('H')).size > 0

print(f'读取完成: {output_json_path}')
print(f'可用字段数: {len(available_fields)}')
print(f'原始匹配数: {raw_match_count}')
print(f'RANSAC 后匹配数: {ransac_match_count}')
print(f'是否存在 H: {has_homography}')


In [ ]:
overview = pd.DataFrame(
    [
        ('JSON 路径', str(output_json_path)),
        ('matcher', matcher),
        ('原始匹配数量', raw_match_count),
        ('RANSAC 后匹配数量', ransac_match_count),
        ('是否存在几何矩阵 H', has_homography),
    ],
    columns=['项目', '值'],
)

display(overview)
print('可用字段列表:')
for field_name in available_fields:
    print(f'  - {field_name}')


In [ ]:
raw_mkpts0 = as_points(pred.get('mkeypoints0_orig'))
raw_mkpts1 = as_points(pred.get('mkeypoints1_orig'))

if len(raw_mkpts0) == 0 or len(raw_mkpts1) == 0:
    print('JSON 中没有可视化所需的原始匹配字段 mkeypoints0_orig / mkeypoints1_orig。')
else:
    raw_conf = as_confidence(pred.get('mconf'), len(raw_mkpts0))
    raw_vis = draw_matches_core(
        raw_mkpts0,
        raw_mkpts1,
        pred['image0_orig'],
        pred['image1_orig'],
        raw_conf,
        titles=['Image 0', 'Image 1'],
        dpi=220,
    )
    show_match_image('原始匹配可视化', raw_vis)


In [ ]:
raw_mkpts0 = as_points(pred.get('mkeypoints0_orig'))
raw_mkpts1 = as_points(pred.get('mkeypoints1_orig'))
ransac_mkpts0 = as_points(pred.get('mmkeypoints0_orig'))
ransac_mkpts1 = as_points(pred.get('mmkeypoints1_orig'))

if len(raw_mkpts0) == 0 or len(raw_mkpts1) == 0:
    print('缺少原始匹配字段，无法做 RANSAC 前后对比。')
elif len(ransac_mkpts0) == 0 or len(ransac_mkpts1) == 0:
    print('当前结果没有 mmkeypoints0_orig / mmkeypoints1_orig，跳过 RANSAC 对比。')
else:
    raw_conf = as_confidence(pred.get('mconf'), len(raw_mkpts0))
    ransac_conf = as_confidence(pred.get('mmconf'), len(ransac_mkpts0))

    raw_vis = draw_matches_core(
        raw_mkpts0,
        raw_mkpts1,
        pred['image0_orig'],
        pred['image1_orig'],
        raw_conf,
        titles=['Image 0', 'Image 1'],
        dpi=180,
    )
    ransac_vis = draw_matches_core(
        ransac_mkpts0,
        ransac_mkpts1,
        pred['image0_orig'],
        pred['image1_orig'],
        ransac_conf,
        titles=['Image 0', 'Image 1'],
        dpi=180,
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(raw_vis)
    axes[0].set_title(f'Raw matches: {len(raw_mkpts0)}')
    axes[0].axis('off')

    axes[1].imshow(ransac_vis)
    axes[1].set_title(f'RANSAC matches: {len(ransac_mkpts0)}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
confidence_series = []

raw_conf = np.asarray(pred.get('mconf')) if pred.get('mconf') is not None else np.asarray([])
if raw_conf.size > 0:
    confidence_series.append(('mconf', raw_conf.reshape(-1).astype(float)))

ransac_conf = np.asarray(pred.get('mmconf')) if pred.get('mmconf') is not None else np.asarray([])
if ransac_conf.size > 0:
    confidence_series.append(('mmconf', ransac_conf.reshape(-1).astype(float)))

if not confidence_series:
    print('当前结果没有 mconf / mmconf，跳过置信度统计可视化。')
else:
    fig, axes = plt.subplots(1, len(confidence_series), figsize=(7 * len(confidence_series), 4))
    if len(confidence_series) == 1:
        axes = [axes]

    stats_rows = []
    for ax, (name, values) in zip(axes, confidence_series):
        ax.hist(values, bins=30, color='#4C78A8', edgecolor='white')
        ax.set_title(f'{name} 分布')
        ax.set_xlabel('confidence')
        ax.set_ylabel('count')

        stats_rows.append({
            '字段': name,
            '数量': int(values.size),
            '均值': float(values.mean()),
            '中位数': float(np.median(values)),
            '最小值': float(values.min()),
            '最大值': float(values.max()),
        })

    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(stats_rows))


In [ ]:
geom_info = pred.get('geom_info', {})

print('H:')
print(format_array(pred.get('H')))

if not geom_info:
    print('当前结果没有 geom_info，可理解为当前 matcher 或当前样本没有产生额外几何结果。')
else:
    print('geom_info 字段摘要:')
    pprint({key: (list(value.shape) if isinstance(value, np.ndarray) else type(value).__name__) for key, value in geom_info.items()})
    print('\ngeom_info 详细内容:')
    for key, value in geom_info.items():
        print(f'\n[{key}]')
        if isinstance(value, np.ndarray):
            print(format_array(value))
        else:
            print(value)
